# 커스텀 Object Detection 모델 학습 코드

| 항목 | 선택 | 이유 |
|---|---|---|
| 아키텍처 | YOLO11n | IMX500 변환 경로(MCT 양자화)가 지원하는 사실상 유일한 실용 모델 |
| 클래스 수 | 1개 | 검출 헤드 크기를 최소화하여 적은 데이터로도 잘 수렴함 |
| 입력 크기 | 320 | IMX500 지원 크기 중 작은 편으로, 드론 실시간 추적에 유리함 |
| 백본 | 앞 10개 층 고정(freeze) | 합성 데이터의 인위적 특징에 과적합되는 현상을 방지함 |

**시작하기 전:** 상단 메뉴 → 런타임 → 런타임 유형 변경 → **T4 GPU** 선택


## 0. GPU 확인


In [ ]:
!nvidia-smi -L
print('위에 Tesla T4 같은 게 안 보이면 런타임 유형을 GPU로 바꾸세요!')

## 1. 설치

학습용 패키지(`ultralytics`)와 IMX500 모델 변환용 패키지를 설치합니다. (약 5분 소요)

> ⚠️ 설치 과정에서 `protobuf` 버전 다운그레이드로 인해 여러 줄의 충돌 경고가 발생할 수 있습니다. 본 노트북에서 사용하지 않는 패키지들이므로 **무시하셔도 됩니다.** 
> 셀 실행이 끝나면 런타임이 **자동으로 재시작**됩니다.
> "세션이 다운되었습니다"라는 메시지 역시 정상적인 현상이므로, 재시작 완료 후 다음 셀부터 이어서 실행하세요.


In [ ]:
!apt-get -qq install -y openjdk-21-jre > /dev/null

!pip install -q ultralytics \
    "model-compression-toolkit>=2.4.1" "edge-mdt-cl<1.1.0" "edge-mdt-tpc>=1.2.0" \
    "pydantic<2.12" "imx500-converter[pt]>=3.17.3"

# 교체된 패키지를 반영하려면 런타임을 한 번 재시작해야 합니다.
import os
os.kill(os.getpid(), 9)

런타임 재시작이 완료되면 아래 셀을 실행하여 설치 상태를 확인한 후 진행해 주세요.


In [ ]:
import ultralytics
ultralytics.checks()

## 2. Roboflow 데이터셋 다운로드

Roboflow에서 라벨링한 데이터셋을 가져옵니다.


In [ ]:
# Roboflow 다운로드 코드에서 복사한 URL 을 붙여넣으세요. (key= 뒷부분까지 전부)
RF_URL = 'https://app.roboflow.com/ds/XXXXXXXX?key=YYYYYYYY'

DATASET_DIR = '/content/roboflow'

!rm -rf {DATASET_DIR} && mkdir -p {DATASET_DIR}
!curl -sL "{RF_URL}" -o /content/roboflow.zip
!unzip -q /content/roboflow.zip -d {DATASET_DIR}
!ls {DATASET_DIR}

### 2-1. 라벨 읽기

데이터가 train / valid / test로 나뉘어 있더라도 **모두 통합하여** 사용합니다.


In [ ]:
import os, glob, yaml
import numpy as np

DATA_YAML_RF = sorted(glob.glob(DATASET_DIR + '/**/data.yaml', recursive=True))[0]
ROOT = os.path.dirname(DATA_YAML_RF)
CLASS_NAME = yaml.safe_load(open(DATA_YAML_RF))['names'][0]

# REAL = [(이미지경로, [(꼭짓점 Nx2(0~1), 폴리곤여부), ...]), ...]
REAL, n_poly, n_box = [], 0, 0
for img_path in sorted(glob.glob(ROOT + '/*/images/*.*')):
    lab_path = img_path.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'
    if not os.path.exists(lab_path):
        continue

    shapes = []
    for line in open(lab_path):
        v = line.split()
        if len(v) < 5:
            continue
        nums = np.array(list(map(float, v[1:])))
        if len(nums) == 4:                       # 박스 -> 사각형 꼭짓점으로 변환
            cx, cy, w, h = nums
            pts = np.array([[cx - w / 2, cy - h / 2], [cx + w / 2, cy - h / 2],
                            [cx + w / 2, cy + h / 2], [cx - w / 2, cy + h / 2]])
            shapes.append((pts, False)); n_box += 1
        elif len(nums) >= 6:                     # 폴리곤
            shapes.append((nums.reshape(-1, 2), True)); n_poly += 1
    if shapes:
        REAL.append((img_path, shapes))

print('클래스 이름 :', CLASS_NAME)
print('실사진      :', len(REAL), '장')
print(f'라벨        : 폴리곤 {n_poly}개 / 박스 {n_box}개')
assert REAL, '라벨이 있는 이미지를 찾지 못했습니다. Roboflow export 포맷이 YOLOv11 인지 확인하세요.'

## 3. 라벨 기반 대상 오려내기

라벨 위치를 기준으로 **대상의 실루엣만** 분리합니다. 라벨 형식에 따라 추출 방식이 다릅니다.


In [ ]:
import cv2
import matplotlib.pyplot as plt

USE_COLOR = True         # 박스 라벨일 때만 사용. False 면 GrabCut 만 씁니다

# OpenCV HSV 기준 (H 는 0~179. 흔히 쓰는 0~360 의 절반입니다)
ORANGE_H     = (3, 28)   # 색상: 빨강(3) ~ 노랑(28) 사이
ORANGE_S_MIN = 70        # 채도: 낮출수록 바랜 주황까지 포함
ORANGE_V_MIN = 60        # 명도: 낮출수록 그늘의 어두운 주황까지 포함

PAD = 0.15               # 라벨 주변 여유 (GrabCut 이 배경 샘플을 볼 수 있도록)


def _largest_blob(mask):
    n, lab, stats, _ = cv2.connectedComponentsWithStats(mask, 8)
    if n <= 1:
        return None
    idx = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
    return np.where(lab == idx, 255, 0).astype(np.uint8)


def cutout(img_path, pts, is_poly):
    """라벨 하나에서 대상을 오려내 (RGB, 알파, 방법이름) 를 반환."""
    bgr = cv2.imread(img_path, cv2.IMREAD_COLOR)
    if bgr is None:
        return None, None, 'read-fail'
    H, W = bgr.shape[:2]

    # 라벨을 픽셀 좌표로 옮기고, 여유를 둔 사각형으로 자른다
    px = pts * [W, H]
    x0, y0 = px.min(0)
    x1, y1 = px.max(0)
    mx, my = (x1 - x0) * PAD / 2, (y1 - y0) * PAD / 2
    X0, Y0 = int(max(0, x0 - mx)), int(max(0, y0 - my))
    X1, Y1 = int(min(W, x1 + mx)), int(min(H, y1 + my))

    crop = bgr[Y0:Y1, X0:X1]
    if crop.size == 0 or min(crop.shape[:2]) < 12:
        return None, None, 'too-small'

    # 긴 변 512 로 정규화 (속도)
    s = min(1.0, 512 / max(crop.shape[:2]))
    if s < 1.0:
        crop = cv2.resize(crop, (int(crop.shape[1] * s), int(crop.shape[0] * s)),
                          interpolation=cv2.INTER_AREA)
    ch, cw = crop.shape[:2]

    if is_poly:
        # 폴리곤이면 외곽선을 그대로 마스크로 사용 (추정 없음)
        poly = ((px - [X0, Y0]) * s).astype(np.int32)
        mask = cv2.fillPoly(np.zeros((ch, cw), np.uint8), [poly], 255)
        how = 'polygon'
    else:
        smooth = cv2.bilateralFilter(crop, 7, 60, 60)
        mask, how = None, ''
        if USE_COLOR:
            m = cv2.inRange(cv2.cvtColor(smooth, cv2.COLOR_BGR2HSV),
                            (ORANGE_H[0], ORANGE_S_MIN, ORANGE_V_MIN), (ORANGE_H[1], 255, 255))
            k = np.ones((3, 3), np.uint8)
            m = cv2.morphologyEx(m, cv2.MORPH_CLOSE, k, iterations=2)
            m = cv2.morphologyEx(m, cv2.MORPH_OPEN,  k, iterations=1)
            m = _largest_blob(m)
            if m is not None and m.sum() / 255 > ch * cw * 0.05:   # 박스의 5% 이상은 채워야 함
                mask, how = m, 'color'

        if mask is None:                                  # GrabCut 대체 경로
            gc = np.full((ch, cw), cv2.GC_PR_BGD, np.uint8)
            my_, mx_ = int(ch * PAD / (1 + PAD)), int(cw * PAD / (1 + PAD))
            gc[my_:ch - my_, mx_:cw - mx_] = cv2.GC_PR_FGD                        # 안쪽은 전경일 가능성
            gc[int(ch * .4):int(ch * .6), int(cw * .4):int(cw * .6)] = cv2.GC_FGD  # 중앙은 확실한 전경
            try:
                cv2.grabCut(crop, gc, None, np.zeros((1, 65), np.float64),
                            np.zeros((1, 65), np.float64), 5, cv2.GC_INIT_WITH_MASK)
                m = np.where((gc == cv2.GC_FGD) | (gc == cv2.GC_PR_FGD), 255, 0).astype(np.uint8)
                mask, how = _largest_blob(m), 'grabcut'
            except cv2.error:
                return None, None, 'grabcut-fail'

    if mask is None or mask.sum() / 255 < ch * cw * 0.02:
        return None, None, 'empty'

    mask = cv2.GaussianBlur(mask, (3, 3), 0)          # 경계를 살짝 부드럽게
    ys, xs = np.where(mask > 16)
    a, b, c, d = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    return cv2.cvtColor(crop[a:b, c:d], cv2.COLOR_BGR2RGB), mask[a:b, c:d], how


PIECES = []            # (이름, RGB, 알파, 방법)
for img_path, shapes in REAL:
    for si, (pts, is_poly) in enumerate(shapes):
        rgb, alpha, how = cutout(img_path, pts, is_poly)
        PIECES.append((f'{os.path.basename(img_path)[:24]}#{si}', rgb, alpha, how))

ok = sum(1 for _, r, _, _ in PIECES if r is not None)
print(f'{ok} / {len(PIECES)} 개 라벨에서 대상을 오려냈습니다.')

### 3-1. 오려내기 결과 확인 — **가장 중요한 단계**

원본 이미지, 잘라낸 모양, 합성 결과를 나란히 비교하여 표시합니다.

- **형태가 온전히 추출된 조각만** 다음 단계에서 사용하세요. 일부가 손상되었거나 배경이 포함된 조각은 제외해야 합니다.
- 폴리곤 라벨을 사용하면 대부분 정상적으로 추출됩니다. 오차가 발생한 항목은 Roboflow에서 폴리곤을 다시 수정하세요.
- 박스 라벨 이용 시 `color` 방식이 대부분 실패한다면 `ORANGE_S_MIN`을 40으로 낮추고, `ORANGE_H` 범위를 `(0, 35)`로 넓혀보세요.


In [ ]:
valid = [(i, n, r, a, h) for i, (n, r, a, h) in enumerate(PIECES) if r is not None]

fig, axes = plt.subplots(len(valid), 3, figsize=(10, 3.2 * len(valid)), squeeze=False)
for row, (i, name, rgb, alpha, how) in zip(axes, valid):
    row[0].imshow(rgb);               row[0].set_title(f'[{i}] {name}  ({how})', fontsize=10)
    row[1].imshow(alpha, cmap='gray');row[1].set_title('mask', fontsize=10)
    h, w = alpha.shape
    chk = ((np.indices((h, w)).sum(0) // 12 % 2) * 40 + 190).astype(np.uint8)[..., None].repeat(3, 2)
    a3 = alpha[..., None] / 255.0
    row[2].imshow((chk * (1 - a3) + rgb * a3).astype(np.uint8))
    row[2].set_title('composited', fontsize=10)
    for ax in row:
        ax.axis('off')
plt.tight_layout(); plt.show()

for i, (n, r, _, how) in enumerate(PIECES):
    if r is None:
        print(f'[{i}] {n}  <- 실패 ({how})')

### 3-2. 사용할 조각 선택

위 단계에서 **정상적으로 오려낸 조각의 번호만** 남기세요. 잘 잘린 조각이 하나만 있어도 학습은 가능하지만, 개수가 많을수록 모델 성능이 향상됩니다.


In [ ]:
# 예: KEEP = [0, 1, 3, 4, 7]   /   None 이면 성공한 것 전부 사용
KEEP = None

ICONS = [(r, a) for i, (n, r, a, h) in enumerate(PIECES)
         if r is not None and (KEEP is None or i in KEEP)]
assert ICONS, '사용할 조각이 없습니다. 3번 셀의 설정을 조정해 다시 실행하세요.'
print('합성에 사용할 조각', len(ICONS), '개  |  크기:', [r.shape[:2] for r, _ in ICONS])

## 4. 배경 이미지 준비

오려낸 조각을 합성할 배경 이미지를 다운로드합니다. (Imagenette 데이터셋, 약 330MB / 1~2분 소요)
13,000여 장의 다채로운 배경 위에 합성을 진행하면 다양한 환경에서도 대상을 정확히 탐지하는 능력이 길러집니다.

> 실전 비행장 사진이 있다면 `/content/backgrounds` 디렉토리에 추가하는 것을 권장합니다.
> 단, **대상이 포함되지 않은 배경 사진만** 넣어야 합니다. 대상이 사진에 포함되어 있으면 라벨 없는 정답으로 인식되어 학습을 방해할 수 있습니다.


In [ ]:
BG_DIR = '/content/backgrounds'
!mkdir -p {BG_DIR}
!wget -q -c https://s3.amazonaws.com/fast-ai-imageclas/imagenette2-320.tgz -O /content/bg.tgz
!tar -xzf /content/bg.tgz -C {BG_DIR}

BG_PATHS = [p for p in glob.glob(BG_DIR + '/**/*.*', recursive=True)
            if p.lower().endswith(('.jpg', '.jpeg', '.png'))]
print('배경 이미지', len(BG_PATHS), '장')
assert len(BG_PATHS) > 100, '배경 다운로드 실패. 셀을 다시 실행하세요.'

## 5. 합성 데이터 생성

이미지를 생성할 때마다 다음 변형 옵션들을 무작위로 적용합니다.

| 변형 | 모사하는 실제 환경 |
|---|---|
| 크기 6~40% | 드론 고도 변화에 따른 거리 차이 |
| 회전 0~360° | 드론 기수 방향(Yaw) 변화 |
| 원근 왜곡 | 기울어진 카메라 시야각 |
| 밝기 · 색조 · 대비 | 실내, 직사광선, 그늘 등 조명 변화 |
| 흐림 · 노이즈 | 비행 중 기체 흔들림 및 저조도 센서 노이즈 |
| 그림자 | 바닥 표면에 생기는 물체 그림자 |
| 화면 밖 걸침 | 프레임 가장자리에 부분적으로 걸린 대상 |
| **흰 종이 받침** | 인쇄물을 종이째 바닥에 놓는 실제 배치 |
| **미끼(distractor)** | 색상만 유사한 다른 물체 |

**흰 종이 받침**(`P_PAPER`) 옵션은 아이콘 뒤에 동일한 각도의 흰색 사각형 배경을 생성합니다.
실제 환경에서 종이에 인쇄하여 바닥에 놓는 방식이라면 이 옵션을 활성화하세요. 라벨 바운딩 박스는 종이를 제외하고 **아이콘 영역에만** 지정됩니다.

**미끼** 옵션은 라벨이 지정되지 않는 주황색 도형 및 색상을 변경한 동일 형태 객체를 의미합니다.
이 과정이 없으면 모델이 객체의 "형태"가 아닌 단순한 "주황색 영역"만 탐지하도록 학습될 수 있습니다.


In [ ]:
# --- 생성량 설정 ---------------------------------------------------
N_TRAIN = 3000     # 합성 학습 이미지 수
IMGSZ   = 320      # 한 변 크기 (학습 / 추론 / 변환 전부 이 값으로 통일)

SCALE_RANGE   = (0.06, 0.40)   # 대상이 화면에서 차지하는 비율 (긴 변 기준)
MAX_INSTANCES = 2              # 한 장에 최대 몇 개 넣을지
P_DISTRACTOR  = 0.5            # 미끼를 넣을 확률
P_EMPTY       = 0.08           # 대상이 없는 배경뿐인 이미지 비율
P_PAPER       = 0.7            # 흰 종이 위에 놓인 것으로 합성할 확률 (0 이면 사용 안 함)
PAPER_RATIO   = (1.15, 2.2)    # 종이가 아이콘보다 몇 배 큰지

In [ ]:
import random, math

rng = random.Random(0)


def _rand_bg(size):
    """배경 한 장을 정사각형으로 잘라 size x size 로 만든다."""
    img = None
    for _ in range(10):
        img = cv2.imread(rng.choice(BG_PATHS))
        if img is not None:
            break
    if img is None:
        img = np.full((size, size, 3), rng.randint(60, 200), np.uint8)

    h, w = img.shape[:2]
    s = rng.randint(int(min(h, w) * 0.5), min(h, w))
    x, y = rng.randint(0, w - s), rng.randint(0, h - s)
    img = cv2.cvtColor(cv2.resize(img[y:y + s, x:x + s], (size, size)), cv2.COLOR_BGR2RGB)

    if rng.random() < 0.5:
        img = img[:, ::-1]
    img = np.clip(img * rng.uniform(0.55, 1.35) + rng.uniform(-25, 25), 0, 255).astype(np.uint8)
    return np.ascontiguousarray(img)


# 조각은 (RGB, 그릴영역 알파, 라벨영역 알파) 세 장으로 다룹니다.
# 흰 종이를 깔면 '그릴영역' 은 종이까지, '라벨영역' 은 아이콘만 가리킵니다.

def _on_paper(rgb, alpha):
    """아이콘 뒤에 흰 종이를 깐 조각을 만든다."""
    h, w = alpha.shape
    PH = max(h + 2, int(h * rng.uniform(*PAPER_RATIO)))          # 세로/가로 비율을 따로 뽑아
    PW = max(w + 2, int(w * rng.uniform(*PAPER_RATIO)))          # 종이 방향도 다양해지도록
    oy, ox = rng.randint(0, PH - h), rng.randint(0, PW - w)      # 종이 안에서의 아이콘 위치

    tone = rng.randint(205, 255)                                  # 종이 밝기 (조명에 따라)
    paper = np.full((PH, PW, 3), tone, np.uint8)
    paper = np.clip(paper.astype(np.int16) + np.random.normal(0, 3, paper.shape), 0, 255).astype(np.uint8)

    a3 = (alpha[..., None] / 255.0)
    paper[oy:oy + h, ox:ox + w] = (paper[oy:oy + h, ox:ox + w] * (1 - a3) + rgb * a3).astype(np.uint8)

    label = np.zeros((PH, PW), np.uint8)
    label[oy:oy + h, ox:ox + w] = alpha
    return paper, np.full((PH, PW), 255, np.uint8), label


def _warp(rgb, paint, label, out_size):
    """크기 조정 + 회전 + 원근 왜곡. out_size 는 '라벨 영역' 의 긴 변 기준."""
    ys, xs = np.where(label > 8)
    if len(xs) == 0:
        return None
    s = out_size / max(ys.max() - ys.min() + 1, xs.max() - xs.min() + 1)
    dsize = (max(2, int(rgb.shape[1] * s)), max(2, int(rgb.shape[0] * s)))
    rgb   = cv2.resize(rgb,   dsize, interpolation=cv2.INTER_AREA)
    paint = cv2.resize(paint, dsize, interpolation=cv2.INTER_AREA)
    label = cv2.resize(label, dsize, interpolation=cv2.INTER_AREA)

    h, w = paint.shape
    pad = int(max(h, w) * 0.75) + 2                     # 회전 시 모서리가 잘리지 않도록
    pads = ((pad, pad), (pad, pad))
    rgb   = np.pad(rgb, pads + ((0, 0),))
    paint = np.pad(paint, pads)
    label = np.pad(label, pads)
    H, W = paint.shape

    M = cv2.getRotationMatrix2D((W / 2, H / 2), rng.uniform(0, 360), 1.0)
    j = min(H, W) * rng.uniform(0.0, 0.16)              # 네 모서리를 밀어 원근을 흉내
    src = np.float32([[0, 0], [W, 0], [W, H], [0, H]])
    dst = src + np.float32([[rng.uniform(-j, j), rng.uniform(-j, j)] for _ in range(4)])
    P = cv2.getPerspectiveTransform(src, dst) @ np.vstack([M, [0, 0, 1]])

    rgb   = cv2.warpPerspective(rgb,   P, (W, H), flags=cv2.INTER_LINEAR)
    paint = cv2.warpPerspective(paint, P, (W, H), flags=cv2.INTER_LINEAR)
    label = cv2.warpPerspective(label, P, (W, H), flags=cv2.INTER_LINEAR)

    ys, xs = np.where(paint > 8)
    if len(xs) == 0 or not (label > 8).any():
        return None
    a, b, c, d = ys.min(), ys.max() + 1, xs.min(), xs.max() + 1
    return rgb[a:b, c:d], paint[a:b, c:d], label[a:b, c:d]


def _jitter(rgb, paint, label):
    """조명 / 색 / 흐림을 조각에 입힌다."""
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV).astype(np.int16)
    hsv[..., 0] = (hsv[..., 0] + rng.randint(-6, 6)) % 180
    hsv[..., 1] = np.clip(hsv[..., 1] * rng.uniform(0.7, 1.2), 0, 255)
    hsv[..., 2] = np.clip(hsv[..., 2] * rng.uniform(0.5, 1.3), 0, 255)
    rgb = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

    k = rng.choice([0, 0, 3, 3, 5])
    if k:
        rgb   = cv2.GaussianBlur(rgb,   (k, k), 0)
        paint = cv2.GaussianBlur(paint, (k, k), 0)
        label = cv2.GaussianBlur(label, (k, k), 0)
    return rgb, paint, label


def _blend(canvas, rgb, alpha, x, y):
    """canvas 의 (x, y) 에 알파 합성. 화면 밖으로 나간 부분은 잘라낸다."""
    H, W = canvas.shape[:2]
    h, w = alpha.shape
    x0, y0, x1, y1 = max(0, x), max(0, y), min(W, x + w), min(H, y + h)
    if x1 <= x0 or y1 <= y0:
        return None

    a = alpha[y0 - y:y1 - y, x0 - x:x1 - x].astype(np.float32) / 255.0
    c = rgb[y0 - y:y1 - y, x0 - x:x1 - x].astype(np.float32)
    roi = canvas[y0:y1, x0:x1].astype(np.float32)
    canvas[y0:y1, x0:x1] = (roi * (1 - a[..., None]) + c * a[..., None]).astype(np.uint8)
    return x0, y0


def _to_canvas(mask, x, y, size):
    """조각 마스크를 캔버스 크기 배열에 올린다. 화면 밖으로 나간 부분은 잘린다."""
    out = np.zeros((size, size), np.uint8)
    h, w = mask.shape
    x0, y0, x1, y1 = max(0, x), max(0, y), min(size, x + w), min(size, y + h)
    if x1 > x0 and y1 > y0:
        out[y0:y1, x0:x1] = mask[y0 - y:y1 - y, x0 - x:x1 - x]
    return out


def _shadow(canvas, paint, x, y):
    """조각 아래에 흐릿한 그림자를 깐다."""
    s = cv2.GaussianBlur(paint, (rng.choice([7, 11, 15]),) * 2, 0)
    off = int(max(paint.shape) * rng.uniform(0.03, 0.12))
    _blend(canvas, np.zeros((*s.shape, 3), np.uint8),
           (s * rng.uniform(0.25, 0.5)).astype(np.uint8),
           x + rng.choice([-off, off]), y + rng.choice([-off, off]))


def _make_piece(size):
    """조각 하나를 골라 종이 받침 / 크기 / 회전 / 조명까지 적용해 돌려준다."""
    rgb, alpha = ICONS[rng.randrange(len(ICONS))]
    if rng.random() < P_PAPER:
        rgb, paint, label = _on_paper(rgb, alpha)
    else:
        rgb, paint, label = rgb, alpha, alpha

    lo, hi = math.log(SCALE_RANGE[0]), math.log(SCALE_RANGE[1])
    target = max(10, int(size * math.exp(rng.uniform(lo, hi))))    # 작은 크기가 더 자주 나오도록
    out = _warp(rgb, paint, label, target)
    return None if out is None else _jitter(*out)


def _distractor(canvas):
    """라벨을 달지 않는 미끼. 색만 비슷하거나, 모양만 같고 색이 다른 것."""
    H = canvas.shape[0]
    if rng.random() < 0.5:
        col = (rng.randint(200, 255), rng.randint(90, 180), rng.randint(0, 70))
        s = rng.randint(int(H * 0.05), int(H * 0.30))
        x, y = rng.randint(0, H - 1), rng.randint(0, H - 1)
        if rng.random() < 0.5:
            cv2.circle(canvas, (x, y), s // 2, col, -1)
        else:
            cv2.rectangle(canvas, (x, y), (x + s, y + rng.randint(s // 3, s)), col, -1)
    else:
        out = _make_piece(H)
        if out is None:
            return
        rgb, paint, _ = out
        hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV).astype(np.int16)
        hsv[..., 0] = (hsv[..., 0] + rng.randint(40, 140)) % 180    # 색을 크게 틀어 다른 물체로
        rgb = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)
        _blend(canvas, rgb, paint, rng.randint(-10, H - 10), rng.randint(-10, H - 10))


def make_sample(size=IMGSZ):
    """합성 이미지 1장과 YOLO 라벨 목록(정규화 cx, cy, w, h)을 반환."""
    canvas = _rand_bg(size)
    labels = []

    if rng.random() < P_DISTRACTOR:
        _distractor(canvas)

    placed = []
    if rng.random() >= P_EMPTY:
        for _ in range(rng.randint(1, MAX_INSTANCES)):
            out = _make_piece(size)
            if out is None:
                continue
            rgb, paint, label = out

            h, w = paint.shape
            x = rng.randint(-w // 3, max(-w // 3 + 1, size - w + w // 3))
            y = rng.randint(-h // 3, max(-h // 3 + 1, size - h + h // 3))

            if rng.random() < 0.4:
                _shadow(canvas, paint, x, y)
            if _blend(canvas, rgb, paint, x, y) is None:
                continue
            placed.append((_to_canvas(paint, x, y, size),
                           _to_canvas(label, x, y, size),
                           int((label > 8).sum())))

    # 화면 밖으로 나갔거나 나중에 올린 조각(특히 흰 종이)에 가려진 부분은 빼고 박스를 만든다
    for i, (_, label, total) in enumerate(placed):
        hidden = np.zeros((size, size), bool)
        for later_paint, _, _ in placed[i + 1:]:
            hidden |= later_paint > 8

        ys, xs = np.where((label > 8) & ~hidden)
        if len(xs) == 0:
            continue
        x0, y0, x1, y1 = xs.min(), ys.min(), xs.max() + 1, ys.max() + 1
        if len(xs) / max(total, 1) < 0.45 or (x1 - x0) < 8 or (y1 - y0) < 8:
            continue          # 너무 가려졌거나 너무 작으면 라벨에서 제외
        labels.append(((x0 + x1) / 2 / size, (y0 + y1) / 2 / size,
                       (x1 - x0) / size, (y1 - y0) / size))

    if rng.random() < 0.5:    # 센서 노이즈
        canvas = np.clip(canvas + np.random.normal(0, rng.uniform(2, 10), canvas.shape),
                         0, 255).astype(np.uint8)
    return canvas, labels


print('합성 함수 준비 완료')

### 5-1. 미리보기

**녹색 바운딩 박스가 대상 위치에 정확히 맞닿아 있는지**, 합성 결과가 자연스러운지 확인하세요.
셀을 여러 번 실행할 때마다 매번 새로운 무작위 샘플이 생성됩니다.


In [ ]:
fig, axes = plt.subplots(3, 5, figsize=(16, 10))
for ax in axes.ravel():
    img, labs = make_sample()
    vis = img.copy()
    for cx, cy, w, h in labs:
        p0 = (int((cx - w / 2) * IMGSZ), int((cy - h / 2) * IMGSZ))
        p1 = (int((cx + w / 2) * IMGSZ), int((cy + h / 2) * IMGSZ))
        cv2.rectangle(vis, p0, p1, (0, 255, 0), 2)
    ax.imshow(vis); ax.axis('off')
plt.tight_layout(); plt.show()

### 5-2. 데이터셋 생성

- **train** : 합성 이미지 3,000장 (약 2~4분 소요)
- **val** : Roboflow 실사진 전체 — 실제 라벨을 사용하므로 **모델의 실제 성능을 정확히 평가할 수 있습니다.**

검증 데이터셋의 오염을 방지하기 위해 실사진은 학습 데이터에 포함하지 않습니다.
(단, 합성 조각의 출처가 해당 실사진들이므로 완벽히 독립적인 평가로 보기는 어렵습니다.
가능하다면 라벨링하지 않은 새로운 사진을 추가 촬영하여 8-1 단계에서 검증해 보세요.)


In [ ]:
import shutil
from tqdm.auto import tqdm

DATASET = '/content/synth'
shutil.rmtree(DATASET, ignore_errors=True)
for split in ('train', 'val'):
    os.makedirs(f'{DATASET}/{split}/images', exist_ok=True)
    os.makedirs(f'{DATASET}/{split}/labels', exist_ok=True)

# train: 합성
for i in tqdm(range(N_TRAIN), desc='합성'):
    img, labs = make_sample()
    cv2.imwrite(f'{DATASET}/train/images/{i:05d}.jpg',
                cv2.cvtColor(img, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, 92])
    with open(f'{DATASET}/train/labels/{i:05d}.txt', 'w') as f:
        f.writelines(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}\n' for cx, cy, w, h in labs)

# val: 실사진 + Roboflow 라벨 (폴리곤이면 외접 박스로 변환)
for i, (img_path, shapes) in enumerate(REAL):
    ext = os.path.splitext(img_path)[1]
    shutil.copy(img_path, f'{DATASET}/val/images/{i:05d}{ext}')
    with open(f'{DATASET}/val/labels/{i:05d}.txt', 'w') as f:
        for pts, _ in shapes:
            x0, y0 = pts.min(0); x1, y1 = pts.max(0)
            f.write(f'0 {(x0 + x1) / 2:.6f} {(y0 + y1) / 2:.6f} {x1 - x0:.6f} {y1 - y0:.6f}\n')

DATA_YAML = f'{DATASET}/data.yaml'
yaml.safe_dump({'path': DATASET, 'train': 'train/images', 'val': 'val/images',
                'nc': 1, 'names': [CLASS_NAME]},
               open(DATA_YAML, 'w'), allow_unicode=True, sort_keys=False)

print(open(DATA_YAML).read())
print('train', len(os.listdir(f'{DATASET}/train/images')), '장 (합성)')
print('val  ', len(os.listdir(f'{DATASET}/val/images')),   '장 (실사진)')

## 6. 학습 설정

**아래 하이퍼파라미터 수치들을 조절하며 실험을 진행해 보세요.**


In [ ]:
EPOCHS = 40          # 합성 데이터는 양이 많아 40 에폭이면 충분합니다
BATCH  = 64          # CUDA out of memory 가 나면 32 또는 16 으로 줄이세요
FREEZE = 10          # 백본 앞 10개 층 고정. 0 이면 전체 학습

BASE_MODEL = 'yolo11n.pt'    # IMX500 은 n 크기 계열만 실용적입니다
RUN_NAME   = 'icon_tracker'

## 7. 모델 학습

T4 GPU 환경 기준 약 **10~20분**이 소요됩니다.

합성 과정에서 이미 강력한 데이터 증강을 적용했으므로, ultralytics의 기본 증강 옵션 중 중복 항목은 수치를 낮추어 적용합니다.
학습 로그에 출력되는 `mAP50` 수치는 **실사진 검증 데이터셋 기준**이므로 모델의 실제 탐지 성능으로 신뢰할 수 있습니다.


In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_MODEL)
model.train(
    data=DATA_YAML,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    freeze=FREEZE,
    name=RUN_NAME,
    patience=15,
    plots=True,
    mosaic=0.5,       # 합성 단계와 중복되는 증강은 낮춥니다
    degrees=0.0,      # 회전은 합성에서 이미 0~360도 적용했습니다
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
)

RUN_DIR = str(model.trainer.save_dir)
BEST = RUN_DIR + '/weights/best.pt'
print('\n학습 완료:', BEST)

## 8. 결과 확인

검증 데이터셋이 **실사진**으로 구성되어 있으므로 측정된 `mAP50` 값이 실제 모델 성능에 해당합니다.
**0.7 이상**의 수치가 기록되면 실제 드론 비행 추적에 활용할 수 있는 수준입니다.


In [ ]:
from IPython.display import Image, display

metrics = YOLO(BEST).val(data=DATA_YAML, imgsz=IMGSZ)
print(f'\n[실사진 검증셋] mAP50 = {metrics.box.map50:.3f}   <- 0.7 이상이면 good')
print(f'[실사진 검증셋] mAP50-95 = {metrics.box.map:.3f}')

display(Image(RUN_DIR + '/results.png', width=900))

### 8-1. 실사진 탐지 결과 시각화

수치 지표보다 시각화된 결과 이미지가 문제점을 더 명확히 보여줍니다. 탐지에 실패한 사진이 있다면 아래 해결 대책을 참고하세요.

| 증상 | 해결 대책 |
|---|---|
| 원거리 대상만 탐지 실패 | `SCALE_RANGE` 하한을 `0.03`으로 낮춘 후 5-2 단계부터 재실행 |
| 특정 배경에서만 탐지 실패 | 대상이 포함되지 않은 해당 비행장 배경 사진을 `/content/backgrounds`에 추가 |
| 유사한 주황색 물체를 오탐지 | `P_DISTRACTOR` 수치를 `0.8`로 올린 후 5-2 단계부터 재실행 |
| 전반적인 신뢰도(Confidence)가 낮음 | `FREEZE = 0`, `EPOCHS = 80`으로 재학습 진행 |
| 바운딩 박스가 대상보다 작게 형성됨 | 3-1 단계에서 조각 일부가 손상된 경우입니다. 해당 조각을 `KEEP` 항목에서 제외 |


In [ ]:
test_imgs = sorted(glob.glob(f'{DATASET}/val/images/*'))
best = YOLO(BEST)

cols = min(5, len(test_imgs))
rows = (len(test_imgs) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows), squeeze=False)
for ax in axes.ravel():
    ax.axis('off')

for ax, p in zip(axes.ravel(), test_imgs):
    r = best.predict(p, imgsz=IMGSZ, conf=0.25, verbose=False)[0]
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    conf = r.boxes.conf.tolist()
    ax.set_title(f'{len(conf)} det / max {max(conf):.2f}' if conf else 'NO DETECTION', fontsize=11)
plt.tight_layout(); plt.show()

## 9. IMX500 전용 모델 변환 (양자화)

IMX500 하드웨어 칩셋은 부동소수점 연산을 지원하지 않으므로 모델을 **8비트 정수형으로 압축**합니다.
`data=` 매개변수로 지정된 이미지 데이터를 기반으로 자동 보정(calibration)이 수행됩니다.

변환 작업은 약 **5~10분** 소요되며, 진행되는 동안 런타임을 중단하지 마세요.

> `Exporting on CPU while CUDA is available...` 경고 메시지는 정상적인 출력 현상입니다.


In [ ]:
model = YOLO(BEST)
export_dir = model.export(format='imx', data=DATA_YAML, imgsz=IMGSZ)
print('\n변환 결과 폴더:', export_dir)

!ls -la {export_dir}

## 10. 다운로드


In [ ]:
from google.colab import files

OUT = '/content/imx500_out'
shutil.rmtree(OUT, ignore_errors=True)
os.makedirs(OUT, exist_ok=True)

shutil.copy(os.path.join(export_dir, 'packerOut.zip'), OUT)
shutil.copy(os.path.join(export_dir, 'labels.txt'), OUT)
shutil.copy(BEST, os.path.join(OUT, 'best.pt'))          # 재학습용 백업

print('클래스 목록 (config.yaml 의 target_class 에 이 이름을 씁니다):')
print(open(os.path.join(OUT, 'labels.txt')).read())

shutil.make_archive(f'/content/{RUN_NAME}_imx500', 'zip', OUT)
files.download(f'/content/{RUN_NAME}_imx500.zip')

## 11. 라즈베리파이 탑재

05-custom-model 문서의 4. 라즈베리파이 탑재 및 모델 설정 을 참고해 라즈베리파이에 모델을 탑재합니다.


---
## 트러블슈팅

| 증상 | 해결 방법 |
|---|---|
| 3-1 단계에서 폴리곤 외곽선이 어긋남 | Roboflow에서 해당 폴리곤을 다시 그리거나 `KEEP` 목록에서 제외하세요. |
| 3-1 단계에서 대상 오려내기 실패 (박스 라벨) | `ORANGE_S_MIN`을 40으로 낮추고 `ORANGE_H` 범위를 `(0, 35)`로 넓히세요. |
| 배경까지 함께 잘려 나감 (박스 라벨) | `ORANGE_S_MIN`을 100 이상으로 올리거나 `USE_COLOR = False`로 설정하여 GrabCut을 활용하세요. |
| 5-1 단계에서 바운딩 박스가 아이콘보다 큼 | 종이 영역까지 라벨에 포함된 경우입니다. 3-1 단계에서 해당 조각을 제외하세요. |
| 종이 받침 없이 바닥에 직접 부착할 예정 | `P_PAPER = 0.0`으로 설정하세요. |
| 실사진 mAP 수치가 낮음 | 학습 배경과 실제 현장 환경의 차이가 큽니다. 대상이 없는 현장 배경 사진을 `/content/backgrounds`에 추가하세요. |
| 주황색 물체를 모두 오탐지함 | `P_DISTRACTOR = 0.8`로 상향 조정한 후 5-2 단계부터 재실행하세요. |
| 지상에서는 작동하나 비행 중 탐지 실패 | `SCALE_RANGE` 하한을 0.03으로 낮추어 원거리 샘플 비중을 늘리세요. |
| CUDA out of memory 발생 | `BATCH` 크기를 32 또는 16으로 변경하세요. |
| export 단계 오류 발생 | 런타임을 재시작한 후 설치 확인 셀부터 다시 순차 실행하세요. |
| AI 카메라 탑재 시에만 성능 저하 | 양자화에 따른 정밀도 손실 현상입니다. `IMGSZ = 640`으로 설정 후 5-2 단계부터 재실행하세요. |
